In [ ]:
pip install faker pandas numpy matplotlib seaborn

In [ ]:
import uuid
import random
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta

# Set up Faker and fix all random seeds so results are repeatable
fake = Faker()
Faker.seed(101)
random.seed(101)
np.random.seed(101)


# 1. CUSTOMERS.CSV

print("Generating customers.csv...")

num_customers = 5000

# Build simple customer IDs like CUST_10000, CUST_10001, ...
cust_ids = []
for i in range(num_customers):
    cust_ids.append(f"CUST_{10000 + i}")

# Generate a fake name for every customer
customer_names = []
for i in range(num_customers):
    customer_names.append(fake.name())

# Generate an email for every customer, but leave about 8% blank
emails = []
for i in range(num_customers):
    if random.random() > 0.08:
        emails.append(fake.email())
    else:
        emails.append(np.nan)

# Generate a country code for every customer
countries = []
for i in range(num_customers):
    countries.append(fake.country_code())

df_customers = pd.DataFrame({
    "customer_id": cust_ids,
    "customer_name": customer_names,
    "email": emails,
    "country": countries
})

# Add 150 duplicate rows
duplicate_rows = df_customers.sample(n=150, random_state=101).copy()
df_customers = pd.concat([df_customers, duplicate_rows], ignore_index=True)
df_customers.to_csv("customers.csv", index=False)



# 2. PRODUCTS.CSV

print("Generating products.csv...")

num_products = 800
categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Beauty', 'Sports']

# Build simple product IDs like PROD_1000, PROD_1001, ...
prod_ids = []
for i in range(num_products):
    prod_ids.append(f"PROD_{1000 + i}")

# Generate a product name for every product
product_names = []
for i in range(num_products):
    product_names.append(fake.catch_phrase())

# Pick a random category for every product
product_categories = []
for i in range(num_products):
    product_categories.append(random.choice(categories))

# Pick a random price for every product
unit_prices = []
for i in range(num_products):
    unit_prices.append(round(random.uniform(4.99, 299.99), 2))

# Mark most products as active, some as inactive
active_flags = []
for i in range(num_products):
    active_flags.append(random.choice([True, True, True, False]))

df_products = pd.DataFrame({
    "product_id": prod_ids,
    "product_name": product_names,
    "category": product_categories,
    "unit_price": unit_prices,
    "active_flag": active_flags
})
df_products.to_csv("products.csv", index=False)



# 3. HELPER FUNCTION: build one batch of orders + order items

def generate_order_batch(num_orders, base_date, add_discount_code=False):
    orders = []
    order_items = []

    regions = ['North', 'South', 'East', 'West', 'Central']
    statuses = ['Completed', 'Completed', 'Completed', 'Returned', 'Cancelled']

    for i in range(num_orders):
        order_id = f"ORD_{uuid.uuid4().hex[:10].upper()}"
        cust_id = random.choice(cust_ids)

        # Put the order at a random time within the given day
        order_ts = base_date + timedelta(
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        order_header = {
            "order_id": order_id,
            "customer_id": cust_id,
            "order_ts": order_ts.strftime('%Y-%m-%d %H:%M:%S'),
            "store_region": random.choice(regions),
            "status": random.choice(statuses)
        }

        # Some batches also get a discount_code column (schema change test)
        discount_code = None
        if add_discount_code:
            discount_code = random.choice(['SAVE10', 'FREESHIP', None, None, 'WELCOME20'])
            order_header["discount_code"] = discount_code

        orders.append(order_header)

        # Decide how many line items this order has (1 to 5, weighted)
        num_items = random.choices([1, 2, 3, 4, 5], weights=[30, 25, 20, 15, 10])[0]
        chosen_products = df_products.sample(n=num_items)

        for _, product in chosen_products.iterrows():
            qty = random.randint(1, 4)
            unit_price = product['unit_price']
            line_total = round(qty * unit_price, 2)

            item = {
                "order_id": order_id,
                "product_id": product['product_id'],
                "quantity": qty,
                "unit_price": unit_price,
                "line_total": line_total
            }

            if add_discount_code:
                item["discount_code"] = discount_code

            order_items.append(item)

    return pd.DataFrame(orders), pd.DataFrame(order_items)



# 4. DAY 1 ORDERS & ORDER ITEMS

print("Generating Day 1 Orders Data...")
day1_date = datetime(2026, 7, 5)
df_orders_d1, df_items_d1 = generate_order_batch(20000, day1_date, add_discount_code=False)

df_orders_d1.to_json("orders_day1.json", orient="records", lines=True)
df_items_d1.to_json("order_items_day1.json", orient="records", lines=True)



# 5. DAY 2 ORDERS & ORDER ITEMS (adds the discount_code column)

print("Generating Day 2 Orders Data (With Schema Evolution)...")
day2_date = datetime(2026, 7, 6)
df_orders_d2, df_items_d2 = generate_order_batch(4000, day2_date, add_discount_code=True)

df_orders_d2.to_json("orders_day2.json", orient="records", lines=True)
df_items_d2.to_json("order_items_day2.json", orient="records", lines=True)



# 6. CLICKSTREAM LOGS

def generate_clickstream(num_events, base_date, filename):
    print(f"Generating clickstream events for {filename}...")
    events = ['view_item', 'add_to_cart', 'remove_from_cart', 'search', 'click_ad', 'checkout_start']
    clickstream = []

    for i in range(num_events):
        event_ts = base_date + timedelta(
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        # About 30% of sessions are "guests" with no customer_id
        if random.random() > 0.3:
            customer_id = random.choice(cust_ids)
        else:
            customer_id = None

        clickstream.append({
            "session_id": str(uuid.uuid4()),
            "customer_id": customer_id,
            "event_timestamp": event_ts.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
            "event_type": random.choice(events),
            "page_url": fake.uri_path(),
            "device_type": random.choice(['Desktop', 'Mobile', 'Mobile', 'Tablet'])
        })

    df_click = pd.DataFrame(clickstream).sort_values(by="event_timestamp")
    df_click.to_json(filename, orient="records", lines=True)


generate_clickstream(15000, day1_date, "clickstream_day1.json")
generate_clickstream(15000, day2_date, "clickstream_day2.json")

print("\nAll RetailFlow datasets generated.")

In [ ]:
import os
import pandas as pd

# Folder where your CSV/JSON files are located
output_dir = os.getcwd()

files_to_profile = [
    ("customers.csv", "csv"),
    ("products.csv", "csv"),
    ("orders_day1.json", "json"),
    ("orders_day2.json", "json"),
    ("order_items_day1.json", "json"),
    ("order_items_day2.json", "json"),
    ("clickstream_day1.json", "json"),
    ("clickstream_day2.json", "json"),
]

profile_registry = {}
null_heatmap_data = {}
report_lines = []

for fname, ftype in files_to_profile:
    path = os.path.join(output_dir, fname)

    if ftype == "csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_json(path, lines=True)

    profile_registry[fname] = df

    report_lines.append(f"FILE: {fname}")
    report_lines.append(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    report_lines.append(f"Total Duplicates: {df.duplicated().sum()}")

    for col in df.columns:
        null_percentage = df[col].isnull().mean() * 100
        null_heatmap_data[f"{fname} -> {col}"] = null_percentage

    column_stats = []
    for col in df.columns:
        null_rate = df[col].isnull().mean() * 100
        dtype = str(df[col].dtype)

        non_nulls = df[col].dropna()
        if pd.api.types.is_numeric_dtype(df[col]) and not non_nulls.empty:
            value_range = f"[{non_nulls.min()} to {non_nulls.max()}]"
        elif not non_nulls.empty:
            value_range = f"[{non_nulls.nunique()} distinct clean values]"
        else:
            value_range = "[All values Null]"

        column_stats.append({
            "Column": col,
            "Dtype": dtype,
            "Null %": f"{null_rate:.2f}%",
            "Range/Cardinality": value_range
        })

    stats_df = pd.DataFrame(column_stats)
    report_lines.append(stats_df.to_string(index=False))
    report_lines.append("-" * 60)

report_text = "\n".join(report_lines)

report_path = os.path.join(output_dir, "data_profile_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Report saved to: {report_path}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


# CHART 1: Order Volume by Day

days = ["Day 1 (July 1)", "Day 2 (July 2)"]
volumes = [
    len(profile_registry["orders_day1.json"]),
    len(profile_registry["orders_day2.json"])
]

fig1, ax1 = plt.subplots(figsize=(6, 4))
sns.barplot(x=days, y=volumes, hue=days, palette="Blues_d", legend=False, ax=ax1)
ax1.set_title("Order Batch Volume Comparison (Ingestion Pipeline)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Total Recorded Header Rows")

for i in range(len(volumes)):
    ax1.text(i, volumes[i] + 300, f"{volumes[i]:,}", ha="center", weight="bold")

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "order_volume_by_day.png"), dpi=150)
plt.close()


# CHART 2: Revenue Distribution

df_items_day1 = profile_registry["order_items_day1.json"]
df_items_day2 = profile_registry["order_items_day2.json"]
df_items = pd.concat([df_items_day1, df_items_day2])

fig2, ax2 = plt.subplots(figsize=(7, 4))
sns.histplot(df_items["line_total"], bins=50, kde=True, color="teal", ax=ax2)
ax2.set_title("Distribution of Line Item Revenue Outliers", fontsize=12, fontweight="bold")
ax2.set_xlabel("Line Item Gross Totals ($)")
ax2.set_xlim(0, 300)  # cut the long tail so the chart looks cleaner

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "revenue_distribution.png"), dpi=150)
plt.close()


# CHART 3: Top Categories by Revenue

df_products = profile_registry["products.csv"]
df_all_items = pd.merge(df_items, df_products, on="product_id", how="left")

category_revenue = df_all_items.groupby("category")["line_total"].sum()
category_revenue = category_revenue.sort_values(ascending=False).reset_index()

fig3, ax3 = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=category_revenue, x="line_total", y="category", hue="category", palette="viridis", legend=False, ax=ax3)
ax3.set_title("Top Product Categories by Total Gross Revenue", fontsize=12, fontweight="bold")
ax3.set_xlabel("Aggregated Total Revenue ($)")
ax3.set_ylabel("Category")
ax3.ticklabel_format(style="plain", axis="x")  # show plain numbers, not 1e6

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "top_categories_revenue.png"), dpi=150)
plt.close()


# CHART 4: Null-Rate Heatmap Across All Files

# Turn the null_heatmap_data dict into a table with File, Column, Null_Percentage
heatmap_rows = []
for field_reference, null_percentage in null_heatmap_data.items():
    file_name, column_name = field_reference.split(" -> ")
    heatmap_rows.append({
        "File": file_name,
        "Column": column_name,
        "Null_Percentage": null_percentage
    })

heatmap_df = pd.DataFrame(heatmap_rows)
pivot_heatmap = heatmap_df.pivot(index="Column", columns="File", values="Null_Percentage")
pivot_heatmap = pivot_heatmap.fillna(0)

fig4, ax4 = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pivot_heatmap,
    annot=True,
    fmt=".1f",
    cmap="Reds",
    cbar_kws={"label": "Null Rate (%)"},
    ax=ax4
)
ax4.set_title("Global Data Quality Heatmap: Missing Value Concentrations", fontsize=12, fontweight="bold")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "null_rate_heatmap.png"), dpi=150)
plt.close()

print(f"All 4 plots successfully compiled and exported to '{charts_dir}/'.")